# 7. Paper figure candidates

Every candidate figure of the planning notes of 9 October 2026: the panels A to F, the verification against ERA5, and the candidate layouts L1–L5 put together from them. Nothing here is the final choice. The point is to see every candidate on the real data and then pick one; notebook 06 keeps the figures already chosen.

Every panel is a function in `extant/plots/paper_panels.py` (`pp`) that draws into a whole figure or into part of one, so the same panel goes into several layouts (`pp.compose`). Each is bound to its data once, as a *block* (`partial(pp.map_block, ...)`), drawn on its own, and reused in the layouts. Every array a panel is given is made here, step by step, from what notebooks 02 and 03 saved (and notebook 01's sea ice): nothing is computed inside a figure.

| | Candidate | From |
|---|---|---|
| A1 | Mean change: the multi-model median with the AR6 hatching | 03 §4 `mean_block` |
| A2 | Mean change: the median, and how many models agree on its sign, per forcing | 03 §4 |
| A3 | Mean change, every model, per forcing | 03 §4 |
| A4 | Observed against historical: ERA5's trend, historical's, and where ERA5 falls among the members | 02 §9 `trend_maps`, part 2 `detections` |
| A5 | Regional time series: each experiment's spread, and ERA5 | 03 §9 `regional_seasonal`, 02 §9 `era5_regional` |
| B1 | Width change: the median with the AR6 hatching, and the change in sea ice | 03 §2 `qrange_change`, 01 §5 |
| B2 | Width change, every model, per forcing | 03 §2 |
| B3 | The regional change at every percentile | 03 §9 `regional_quantile_curve` |
| C1 | Probability ratios of warm and cold 1-in-20 seasons, every model | 03 §10 `probability_ratio_regional` |
| C2 | The same, with the ratios the mean alone would give | 03 §10 |
| C3 | Probability-ratio maps | 03 §10 `probability_ratio` |
| R1 | The distributions themselves, as ridgelines: every forcing, each region and season | 03 §9 `regional_seasonal` |
| R2 | The same for every model, historical | 03 §9 |
| D1–D4 | Sea ice and the width | 01 §5 (JASMIN), 03 §2, §9 |
| E1–E5 | The SAM and the mean | 03 §12 |
| F1 | Regional synthesis: every forcing, the sum of the single forcings, and historical | 03 §6, §11 |
| F2 | Additivity: historical minus the sum of the single forcings | 03 §6, §11 |
| V1–V5 | Verification against ERA5 (Suarez-Gutierrez et al., 2021) | 02 §5–9 |
| S | Supplementary: every model's SAM regression and probability ratios | 03 §10, §12 |
| L1–L5 | The candidate layouts | all of the above |

**Conventions.** Rows are forcings and columns seasons, as in notebooks 04–06. A multi-model map is the median across the models, with the IPCC AR6 advanced approach to agreement (`config.SIGNIFICANT_THRESHOLD`, `config.SIGN_THRESHOLD`): no hatching where at least two-thirds of the models have a significant change and at least 80% of them agree on its sign (a robust change); diagonal lines where fewer than two-thirds are significant; crossed lines where they are but do not agree on the sign. Each model's significance is its own test against its own internal variability (notebook 03). The number of models is in each panel's corner. Print style (`fr.PAPER`), a page (183 mm) wide, with the caption as the markdown above each figure; every figure is saved as a PDF and a PNG in `figures/<variable>/candidates/`.

**On the sample** everything runs, but on 3×3 cells and two models, so the maps are dots, the SAM index is a made-up stand-in (notebook 03, section 12), and the sea-ice panels (JASMIN) are left out.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import config, datatree, era5_evaluation as ev, observed_change as oc, paths, response_change as rc
from extant import significance as sig, storage
from extant.logger import setup_logging
from extant.plots import era5_evaluation as evp, forced_response as fr, paper_panels as pp
from extant.stats import fast_kde_xr, nan_quantile

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units

#(t): The forcings in the order of every figure, and the seasons
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The tests' significance level, and the AR6 agreement thresholds
ALPHA = 0.05
SIGNIFICANT_THRESHOLD = config.SIGNIFICANT_THRESHOLD
SIGN_THRESHOLD = config.SIGN_THRESHOLD

#(t): The region of the regional panels, and the sea-ice region of D2-D4 (observed_change.REGIONS)
REGION = 'Antarctica (90–65°S)'
ICE_REGION = 'Southern Ocean (65–50°S)'

#(t): The probability ratios shown: of 1-in-RETURN_PERIOD seasons
RETURN_PERIOD = 20

#(t): The first year of the time series (A5)
PLUME_START = 1950

#(t): The final years compared with hist-nat (notebook 03): 1993-2013
WINDOW = config.WINDOW

#(t): The width of the smoothing kernel of the ridgelines (R1, R2), in °C
RIDGE_BANDWIDTH = 0.3

#(t): Print style, a page wide
STYLE = fr.PAPER
WIDTH = fr.PAGE_WIDTH


def result_path(name):
    #(t): Where notebooks 02 and 03 saved a result: results/<variable>/<run>/<name>.zarr
    return paths.result(name, VARIABLE, RUN)

In [ ]:
#(t): The colour scales: notebook 04's, and those of the probability ratio, the trends, ERA5's percentile, the change
#(t): in variance and the SAM regression
scales = fr.colour_scales(VARIABLE, alpha=ALPHA)
ratio_scale = pp.ratio_scale(f'Probability ratio of a 1-in-{RETURN_PERIOD} season')
trend_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 2), 'RdBu_r', f'Trend ({UNITS} per decade)')
percentile_scale = pp.percentile_scale()
variance_scale = fr.Scale(np.round(np.arange(-2, 2.01, 0.4), 2), 'RdBu_r', f'Change in variance ({UNITS}²)')
regression_scale = fr.Scale(np.round(np.arange(-1.5, 1.51, 0.25), 2), 'RdBu_r', f'{UNITS} per unit of the SAM index')

## Open the saved results

Run **one** of the next two sections. Both leave `RUN`, which says which saved results to open, and `CANDIDATE_DIR`, where the figures go.

### Sample data

In [ ]:
RUN = 'sample'
CANDIDATE_DIR = paths.figure('candidates_sample', VARIABLE)
CANDIDATE_DIR

### Full data (JASMIN)

In [ ]:
RUN = 'full'
CANDIDATE_DIR = paths.figure('candidates', VARIABLE)
CANDIDATE_DIR

### The saved results

In [ ]:
#(t): What notebook 03 saved
mean_block_ds = storage.open_dataset(result_path('mean_block'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
own_changes_da = storage.open_dataarray(result_path('own_changes'), load=True)
own_variance_changes_da = storage.open_dataarray(result_path('own_variance_changes'), load=True)
pr_ds = storage.open_dataset(result_path('probability_ratio'), load=True)
regional_pr_ds = storage.open_dataset(result_path('probability_ratio_regional'), load=True)
regional_quantile_curve_da = storage.open_dataarray(result_path('regional_quantile_curve'), load=True)
regional_season_tree = storage.open_tree(result_path('regional_seasonal'), load=True)
sam_change_da = storage.open_dataarray(result_path('sam_change'), load=True)
sam_regression_da = storage.open_dataarray(result_path('sam_regression'), load=True)
sam_split_ds = storage.open_dataset(result_path('sam_split'), load=True)
sam_phase_width_ds = storage.open_dataset(result_path('sam_phase_width'), load=True)

In [ ]:
#(t): What notebook 02 saved
trend_maps_ds = storage.open_dataset(result_path('trend_maps'), load=True)
detections_ds = storage.open_dataset(result_path('detections'), load=True)
era5_regional_da = storage.open_dataarray(result_path('era5_regional'), load=True)
regional_ranks_ds = storage.open_dataset(result_path('regional_ranks'), load=True)
regional_consistency_ds = storage.open_dataset(result_path('regional_consistency'), load=True)
spatial_table_ds = storage.open_dataset(result_path('spatial_table'), load=True)
eval_map_table_ds = storage.open_dataset(result_path('eval_map_table'), load=True)

In [ ]:
#(t): The models: every count runs up to this many
N_MODELS = mean_block_ds.sizes['model']
list(mean_block_ds['model'].values)

### Sea ice

The change in sea-ice concentration (notebook 01, section 5), for B1 and D1–D4. Run **one** of the next two sections.

#### Without sea ice

In [ ]:
ice_change_da = None
ice_contours = None

#### With sea ice (JASMIN)

In [ ]:
#(t): Each experiment's ensemble-mean concentration over its final years, on (model, experiment, season, lat, lon), %
ice_climatology_da = storage.open_dataarray(paths.result('climatology', 'siconc', 'full'), load=True)
ice_climatology_da

In [ ]:
#(t): The change: each experiment minus hist-nat
ice_change_da = ice_climatology_da - ice_climatology_da.sel(experiment='hist-nat')
ice_change_da

In [ ]:
#(t): Its multi-model median as lines over the width maps (B1): ±5 and ±15%, dashed where ice is lost
ice_contours = pp.Contours(ice_change_da.median('model'), levels=(-15, -5, 5, 15),
                           label='Change in sea-ice concentration (±5, ±15%; dashed: loss)')

## The models together

The AR6 advanced approach (`sig.model_agreement`) for each change: the median across models, how many of them have data, the share that agree on the sign of the median, and the three exclusive classes, `robust`, `conflicting` and `no_signal`. A model's change counts as significant by its own test in notebook 03: the member-block permutation test for the mean, the hist-nat bootstrap for the width, the bootstrap of members for a probability ratio.

In [ ]:
#(t): The mean
mean_agreement_ds = sig.model_agreement(mean_block_ds['mean_change'], mean_block_ds['pvalue'] < ALPHA,
                                        sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)
mean_agreement_ds

In [ ]:
#(t): The width, Q95 - Q05 of the internal variability
width_agreement_ds = sig.model_agreement(qrange_change_ds['qrange_change'], qrange_change_ds['qrange_pvalue'] < ALPHA,
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The probability ratios of 1-in-RETURN_PERIOD seasons as log2 (+1 doubles the chance; a ratio of 0 kept finite)
log_ratio_da = np.log2(pr_ds['pr'].sel(return_period=RETURN_PERIOD).clip(min=2.0 ** -10))
log_ratio_da

In [ ]:
#(t): and their agreement
ratio_agreement_ds = sig.model_agreement(log_ratio_da, pr_ds['pr_significant'].sel(return_period=RETURN_PERIOD),
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The AR6 hatching of each: diagonal lines where no robust change, crossed lines where conflicting
mean_hatches = [pp.Hatch(mean_agreement_ds[name], *pp.AGREEMENT[name]) for name in pp.AGREEMENT]
width_hatches = [pp.Hatch(width_agreement_ds[name], *pp.AGREEMENT[name]) for name in pp.AGREEMENT]
ratio_hatches = [pp.Hatch(ratio_agreement_ds[name], *pp.AGREEMENT[name]) for name in pp.AGREEMENT]

In [ ]:
#(t): The number of models in each panel
mean_n_models_da = mean_agreement_ds['n_models'].max(['lat', 'lon'])
width_n_models_da = width_agreement_ds['n_models'].max(['lat', 'lon'])
ratio_n_models_da = ratio_agreement_ds['n_models'].max(['lat', 'lon'])
mean_n_models_da.to_pandas()

In [ ]:
#(t): How much of the area south of 60°S each forcing robustly changes (%)
robust_area_df = pd.DataFrame({
    'mean': (100 * sig.area_mean(mean_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
    'width': (100 * sig.area_mean(width_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
}).unstack('season').round(0)
robust_area_df

## A. The mean

**A1. The forced change in the mean.** Multi-model median change in the seasonal mean temperature, 1993–2013, against hist-nat over the same years, for each forcing (rows) and season (columns). Hatching after IPCC AR6: diagonal lines, fewer than two-thirds of the models have a significant change (member-block permutation test, p < 0.05); crossed lines, at least two-thirds do but fewer than 80% agree on its sign; none, a robust change. n: the number of models.

In [ ]:
a1_block = partial(pp.map_block, mean_agreement_ds['median'], scales.change, row_dim='experiment', rows=EXPERIMENTS,
                   columns=SEASONS, hatches=mean_hatches, notes=mean_n_models_da, style=STYLE)
panels = a1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'A1_mean_agreement')

**A2. The mean change, and how many models agree on its sign.** For each forcing (one figure each): the multi-model median change (top) and the number of models whose change has the sign of the median (bottom).

In [ ]:
#(t): How many models agree on the sign of the median
mean_sign_count_da = (mean_agreement_ds['sign_agreement'] * mean_agreement_ds['n_models']).round()
mean_sign_count_da

In [ ]:
#(t): One figure per forcing: the median, then the count
count_scale = fr.majority_scale(N_MODELS, 'Models agreeing on the sign of the median')
a2_panels = {}
for experiment in EXPERIMENTS:
    median_block = partial(pp.map_block, mean_agreement_ds['median'].sel(experiment=experiment), scales.change,
                           columns=SEASONS, style=STYLE)
    count_block = partial(pp.map_block, mean_sign_count_da.sel(experiment=experiment), count_scale, columns=SEASONS,
                          style=STYLE)
    a2_panels[experiment] = pp.compose([[median_block], [count_block]], heights=[2.5, 2.5], style=STYLE)

In [ ]:
for experiment, panels in a2_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'A2_mean_and_sign_{experiment}')

**A3. The mean change in every model.** For each forcing (one figure each): every model's change (rows) in each season. Dots: not significant in that model (member-block permutation test, p ≥ 0.05).

In [ ]:
#(t): Not significant in a model, where it has data
mean_not_significant_da = ~(mean_block_ds['pvalue'] < ALPHA) & mean_block_ds['mean_change'].notnull()

In [ ]:
a3_panels = {experiment: pp.map_block(mean_block_ds['mean_change'].sel(experiment=experiment), scales.change,
                                      row_dim='model', columns=SEASONS, style=STYLE, tag_start=0,
                                      hatches=[pp.Hatch(mean_not_significant_da.sel(experiment=experiment),
                                                        *pp.NOT_SIGNIFICANT)])
             for experiment in EXPERIMENTS}

In [ ]:
for experiment, panels in a3_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'A3_mean_every_model_{experiment}')

**A4. The observed change against historical.** (a–d) ERA5's trend over the years ERA5 and the models share (1979–2013) and (e–h) the multi-model median of historical's ensemble-mean trend over the same years. (i–l) ERA5's percentile among historical's members (each model's 1979–2013 trends, the median across models): central colours, ERA5 behaves like a member; dark blue or red, ERA5's trend is below or above 95% of the members. Dots: ERA5's trend is outside hist-nat's 5–95% range in at least two-thirds of the models (detected).

In [ ]:
#(t): ERA5's trend and historical's multi-model median trend, along a 'source' dim
observed_trend_da = xr.concat([trend_maps_ds['era5_trend'],
                               trend_maps_ds['model_trend'].sel(experiment='historical', drop=True).median('model')],
                              dim=xr.Variable('source', ['ERA5', 'historical (median)']))
observed_trend_da

In [ ]:
#(t): ERA5's percentile among historical's members, the median across models
era5_percentile_da = detections_ds['historical_percentile'].median('model')

In [ ]:
#(t): Where ERA5's trend is outside hist-nat's range (detected: classes 1 and 2) in at least two-thirds of the models
n_detection_models_da = detections_ds['detection_class'].notnull().sum('model')
n_detected_da = detections_ds['detection_class'].isin([1, 2]).sum('model')
detected_da = (n_detected_da >= SIGNIFICANT_THRESHOLD * n_detection_models_da) & (n_detection_models_da > 0)

In [ ]:
a4_trend_block = partial(pp.map_block, observed_trend_da, trend_scale, row_dim='source', columns=SEASONS,
                         style=STYLE)
a4_percentile_block = partial(pp.map_block, era5_percentile_da, percentile_scale, columns=SEASONS, style=STYLE,
                              hatches=[pp.Hatch(detected_da, '....', "Detected: ERA5 outside hist-nat's 5–95% in "
                                                                     "at least 2/3 of the models")])
panels = pp.compose([[a4_trend_block], [a4_percentile_block]], heights=[4.4, 2.6], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'A4_observed_vs_historical')

**A5. Regional time series.** Seasonal mean temperature averaged over the region, as anomalies from historical's 1979–2013 mean in each model (ERA5: from its own 1979–2013 mean). Bands: the 5–95% range of every member of every model, pooled, for each experiment; lines: their medians; black: ERA5.

In [ ]:
#(t): Every member's regional means (notebook 03, section 9) as one Dataset, members padded with NaN where models differ
regional_season_ds = datatree.tree_to_dataset(regional_season_tree, dims=['model', 'experiment'])
regional_season_ds

In [ ]:
#(t): Each model's historical ensemble mean over 1979-2013: the baseline of its anomalies
historical_baseline_da = regional_season_ds[VARIABLE].sel(experiment='historical', year=slice(1979, 2013)).mean(['member', 'year'])
historical_baseline_da

In [ ]:
#(t): Every member's anomaly from it, every experiment
regional_anomaly_da = regional_season_ds[VARIABLE] - historical_baseline_da

In [ ]:
#(t): Every member of every model pooled
pooled_anomaly_da = regional_anomaly_da.stack(pooled=['model', 'member'])

In [ ]:
#(t): The pooled 5%, 50% and 95% of each experiment in each year
plume_da = nan_quantile(pooled_anomaly_da, [0.05, 0.5, 0.95], 'pooled')
plume_da

In [ ]:
#(t): ERA5's anomalies from its own 1979-2013 mean
era5_anomaly_da = era5_regional_da - era5_regional_da.sel(year=slice(1979, 2013)).mean('year')

In [ ]:
a5_block = partial(pp.plumes, plume_da.sel(region=REGION), era5_anomaly_da.sel(region=REGION),
                   experiments=['hist-nat', *EXPERIMENTS], columns=SEASONS, units=UNITS, start_year=PLUME_START,
                   style=STYLE)
panels = a5_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'A5_regional_time_series')

## B. The width

**B1. The forced change in width.** Multi-model median change in the width Q95 − Q05 of the internal variability (each member minus its ensemble's forced response), 1993–2013 against hist-nat's whole record. Hatching as in A1 (hist-nat bootstrap, p < 0.05). Lines (with the sea ice, D1): the multi-model median change in sea-ice concentration, ±5 and ±15%, dashed where ice is lost.

In [ ]:
b1_block = partial(pp.map_block, width_agreement_ds['median'], scales.spread_change, row_dim='experiment',
                   rows=EXPERIMENTS, columns=SEASONS, hatches=width_hatches, notes=width_n_models_da,
                   contours=ice_contours, style=STYLE)
panels = b1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'B1_width_agreement')

**B2. The change in width in every model.** As A3, for the width. Dots: not significant (hist-nat bootstrap, p ≥ 0.05).

In [ ]:
width_not_significant_da = ~(qrange_change_ds['qrange_pvalue'] < ALPHA) & qrange_change_ds['qrange_change'].notnull()

In [ ]:
b2_panels = {experiment: pp.map_block(qrange_change_ds['qrange_change'].sel(experiment=experiment),
                                      scales.spread_change, row_dim='model', columns=SEASONS, style=STYLE,
                                      tag_start=0,
                                      hatches=[pp.Hatch(width_not_significant_da.sel(experiment=experiment),
                                                        *pp.NOT_SIGNIFICANT)])
             for experiment in EXPERIMENTS}

In [ ]:
for experiment, panels in b2_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'B2_width_every_model_{experiment}')

**B3. The change at every percentile.** The change in every 5th percentile, 1993–2013 against hist-nat's whole record, averaged over each region (rows), for each forcing: the multi-model median (lines) and the models' range (shading). A flat line is a shift of the whole distribution; a line falling to the right, the cold tail warming more than the warm tail (a narrowing).

In [ ]:
b3_block = partial(pp.quantile_curves, regional_quantile_curve_da, experiments=EXPERIMENTS, rows=list(oc.REGIONS),
                   columns=SEASONS, units=UNITS, style=STYLE)
panels = b3_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'B3_quantile_curves')

## C. Extremes: probability ratios

**C1. How much more likely a warm or a cold season.** Probability ratio of a season warmer than hist-nat's 1-in-20 level (top) or colder than its 1-in-20 cold level (bottom), regional means, 1993–2013, each forcing against hist-nat. Markers: each model (shape), with its 95% bootstrap range (whole members resampled); bars: the multi-model median. Log scale; a ratio of 0 sits on the bottom edge.

In [ ]:
#(t): The region's ratios of 1-in-RETURN_PERIOD seasons
regional_ratio_ds = regional_pr_ds.sel(region=REGION, return_period=RETURN_PERIOD)
regional_ratio_ds[['pr', 'pr_lower', 'pr_upper', 'pr_mean_only']].to_dataframe().unstack('season').round(2)

In [ ]:
c1_block = partial(pp.pr_points, regional_ratio_ds, experiments=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = c1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'C1_probability_ratios')

**C2. The mean alone, and the rest.** As C1, with open markers for the ratio a shift of the mean alone would give (hist-nat moved by the change in the mean). The gap between a filled and an open marker is what the change in width and shape adds; it matters most for the cold tail.

In [ ]:
c2_block = partial(pp.pr_points, regional_ratio_ds, experiments=EXPERIMENTS, columns=SEASONS, mean_only=True,
                   style=STYLE)
panels = c2_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'C2_probability_ratios_mean_only')

**C3. Probability-ratio maps.** Multi-model median probability ratio of a warm (one figure) and a cold (the other) 1-in-20 season, 1993–2013 against hist-nat. Orange: more likely; purple: less likely. Hatching as in A1, with significance from the bootstrap (the 95% range of a model's ratio excludes 1).

In [ ]:
c3_blocks = {tail: partial(pp.map_block, ratio_agreement_ds['median'].sel(tail=tail), ratio_scale,
                           row_dim='experiment', rows=EXPERIMENTS, columns=SEASONS, style=STYLE,
                           notes=ratio_n_models_da.sel(tail=tail),
                           hatches=[pp.Hatch(hatch.mask.sel(tail=tail), hatch.pattern, hatch.label)
                                    for hatch in ratio_hatches])
             for tail in ('warm', 'cold')}
c3_panels = {tail: block(tag_start=0) for tail, block in c3_blocks.items()}

In [ ]:
for tail, panels in c3_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'C3_probability_ratio_maps_{tail}')

## R. The distributions themselves: ridgelines

Every number in A–C summarises a distribution; these show the distributions. Each ridge is the distribution of one experiment's seasonal means over 1993–2013, as anomalies from its model's hist-nat mean over the same years, so hist-nat is centred on 0 in every model. Ridges are coloured by their mean, blue through purple to red, with the mean written beside them. The dotted lines are hist-nat's 1-in-20 cold and warm levels, so the share of a ridge beyond them is what the probability ratios (C1, C2) count: a ridge that moves right puts more beyond the warm line, and one that also narrows puts less beyond the cold line than the shift alone would.

In [ ]:
#(t): The final WINDOW years of every member's regional means (regional_season_ds: A5)
final_regional_da = regional_season_ds[VARIABLE].isel(year=slice(-WINDOW, None))
final_regional_da

In [ ]:
#(t): Each model's hist-nat mean over the same years: what its anomalies are measured from
hist_nat_final_mean_da = final_regional_da.sel(experiment='hist-nat').mean(['member', 'year'])
hist_nat_final_mean_da

In [ ]:
#(t): Every member-year as an anomaly from its model's hist-nat
final_anomaly_da = final_regional_da - hist_nat_final_mean_da

In [ ]:
#(t): The x axis of the densities: a little beyond the smallest and largest anomalies (0.1 and 99.9 percentiles)
ridge_low, ridge_high = np.nanpercentile(final_anomaly_da.values, [0.1, 99.9])
ridge_x_arr = np.linspace(np.floor(ridge_low) - 1, np.ceil(ridge_high) + 1, 241)
ridge_x_arr[[0, -1]]

In [ ]:
#(t): The density of each experiment, every member of every model pooled, on (experiment, season, region, x)
pooled_density_da = fast_kde_xr(final_anomaly_da, ridge_x_arr, dim=['model', 'member', 'year'],
                                bandwidth=RIDGE_BANDWIDTH)
pooled_density_da

In [ ]:
#(t): The mean of each: its colour, and the number beside it
pooled_mean_da = final_anomaly_da.mean(['model', 'member', 'year'])
pooled_mean_da.sel(region=REGION).to_pandas().round(2)

In [ ]:
#(t): hist-nat's 1-in-RETURN_PERIOD cold and warm levels, every member of every model pooled: the dotted lines
pooled_levels_da = nan_quantile(final_anomaly_da.sel(experiment='hist-nat'),
                                [1 / RETURN_PERIOD, 1 - 1 / RETURN_PERIOD], ['model', 'member', 'year'])
pooled_levels_da.sel(region=REGION).to_pandas().round(2)

In [ ]:
#(t): Quick look: the densities in REGION, one line per experiment, one panel per season
pooled_density_da.sel(region=REGION).plot.line(x='x', hue='experiment', col='season');

**R1. How each forcing changes the whole distribution.** Distributions of seasonal mean temperature over the region, 1993–2013, for hist-nat and each forcing (ridges), as anomalies from each model's hist-nat mean over the same years, every member of every model pooled; one panel per season. Colour and number: each distribution's mean. Dotted: hist-nat's 1-in-20 cold and warm levels. One figure per region.

In [ ]:
r1_blocks = {region: partial(pp.ridgelines, pooled_density_da.sel(region=region), pooled_mean_da.sel(region=region),
                             ridges=['hist-nat', *EXPERIMENTS], columns=SEASONS,
                             lines=pooled_levels_da.sel(region=region), xlabel=f'Anomaly from hist-nat ({UNITS})',
                             colorbar_label=f'Mean change ({UNITS})', style=STYLE)
             for region in oc.REGIONS}
r1_panels = {region: block(tag_start=0) for region, block in r1_blocks.items()}

In [ ]:
for region, panels in r1_panels.items():
    fr.save(panels, CANDIDATE_DIR, f"R1_ridgelines_{region.split(' (')[0].replace(' ', '_')}")

**R2. Every model.** As R1 for historical, every model on its own (each from its own hist-nat), with every model pooled at the top: how far the models agree on the shift and on the width.

In [ ]:
#(t): historical's anomalies
historical_anomaly_da = final_anomaly_da.sel(experiment='historical', drop=True)

In [ ]:
#(t): Each model's own density, on (model, season, region, x)
model_density_da = fast_kde_xr(historical_anomaly_da, ridge_x_arr, dim=['member', 'year'], bandwidth=RIDGE_BANDWIDTH)

In [ ]:
#(t): With every model pooled (R1's historical ridge) first, as 'All models'
all_models_density_da = pooled_density_da.sel(experiment='historical', drop=True).expand_dims(model=['All models'])
historical_density_da = xr.concat([all_models_density_da, model_density_da], dim='model')
historical_density_da

In [ ]:
#(t): The means, in the same order
all_models_mean_da = pooled_mean_da.sel(experiment='historical', drop=True).expand_dims(model=['All models'])
historical_mean_da = xr.concat([all_models_mean_da, historical_anomaly_da.mean(['member', 'year'])], dim='model')
historical_mean_da.sel(region=REGION).to_pandas().round(2)

In [ ]:
r2_block = partial(pp.ridgelines, historical_density_da.sel(region=REGION), historical_mean_da.sel(region=REGION),
                   ridge_dim='model', columns=SEASONS, lines=pooled_levels_da.sel(region=REGION),
                   xlabel=f'Anomaly from hist-nat ({UNITS})', colorbar_label=f'Mean change ({UNITS})', style=STYLE)
panels = r2_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'R2_ridgelines_every_model_historical')

## D. Sea ice and the width

The expectation: where sea ice is lost, the open ocean holds the surface temperature close to freezing in winter less often, so the cold tail warms most and the distribution narrows; strongest in JJA and SON, weak in DJF. **D1** is B1 with the sea-ice change drawn (run *With sea ice*). D2 and D4 need the sea ice (JASMIN); D3 does not.

**D3. The change at every percentile in the sea-ice region.** As B3, for `ICE_REGION`: expected to fall to the right (the cold tail warming most) where ice is lost.

In [ ]:
d3_block = partial(pp.quantile_curves, regional_quantile_curve_da, experiments=EXPERIMENTS, rows=[ICE_REGION],
                   columns=SEASONS, units=UNITS, style=STYLE)
panels = d3_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'D3_quantile_curves_sea_ice_region')

### D2 and D4 (JASMIN)

**D2. Sea-ice loss against the change in width.** Each point is one model (marker) and forcing (colour): the change in sea-ice concentration against the change in width, both averaged over `ICE_REGION`, one panel per season; r is the correlation over all the points. **D4** is the same against the change in the mean, as a cross-check.

In [ ]:
#(t): The change in sea-ice concentration averaged over the region, on (model, experiment, season)
ice_regional_change_da = oc.regional_mean(ice_change_da, ICE_REGION)
ice_regional_change_da.sel(experiment=EXPERIMENTS).to_dataframe().unstack('season').round(1)

In [ ]:
#(t): The change in width over the same region
width_ice_region_da = oc.regional_mean(qrange_change_ds['qrange_change'], ICE_REGION)

In [ ]:
#(t): and in the mean
mean_ice_region_da = oc.regional_mean(mean_block_ds['mean_change'], ICE_REGION)

In [ ]:
d2_block = partial(pp.scatter_by_season, ice_regional_change_da, width_ice_region_da, 'Δ sea ice (%)',
                   f'Δ(Q95 − Q05) ({UNITS})', experiments=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = d2_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'D2_sea_ice_vs_width')

In [ ]:
d4_block = partial(pp.scatter_by_season, ice_regional_change_da, mean_ice_region_da, 'Δ sea ice (%)',
                   f'Δ mean ({UNITS})', experiments=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = d4_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'D4_sea_ice_vs_mean')

## E. The SAM and the mean

From notebook 03, section 12. On the sample the index is made up, so these only show that the figures work.

**E1. The forced change in the SAM.** Each forcing's change in the SAM index, 1993–2013 against hist-nat: the multi-model median (bars) and every model (markers).

In [ ]:
sam_change_da.to_dataframe().unstack('season').round(2)

In [ ]:
e1_block = partial(pp.forcing_bars, {'Δ SAM index': sam_change_da}, bars=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = e1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'E1_sam_change')

**E2. The change in the mean that goes with the SAM.** For each forcing (one figure each), the multi-model median of: the change in the mean (top), its SAM-congruent part, the regression on the SAM times the forced change in the SAM (middle), and the rest (bottom).

In [ ]:
#(t): The multi-model median of each part, along a 'part' dim
sam_parts_da = sam_split_ds.median('model').to_dataarray('part')
sam_parts_da

In [ ]:
SAM_PART_LABELS = {'mean_change': 'Δ mean', 'congruent': 'SAM-congruent', 'residual': 'Residual'}
e2_blocks = {experiment: partial(pp.map_block, sam_parts_da.sel(experiment=experiment), scales.change,
                                 row_dim='part', rows=list(SAM_PART_LABELS), row_labels=SAM_PART_LABELS,
                                 columns=SEASONS, title=experiment, style=STYLE)
             for experiment in EXPERIMENTS}
e2_panels = {experiment: block(tag_start=0) for experiment, block in e2_blocks.items()}

In [ ]:
for experiment, panels in e2_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'E2_sam_split_{experiment}')

**E3. The change in the SAM against the change in the mean** over the region, a point per model and forcing. **E5** is the same against the change in width.

In [ ]:
#(t): The change in the mean and in the width averaged over REGION, on (model, experiment, season)
mean_region_da = oc.regional_mean(mean_block_ds['mean_change'], REGION)
width_region_da = oc.regional_mean(qrange_change_ds['qrange_change'], REGION)

In [ ]:
e3_block = partial(pp.scatter_by_season, sam_change_da, mean_region_da, 'Δ SAM index', f'Δ mean ({UNITS})',
                   experiments=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = e3_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'E3_sam_vs_mean')

**E4. The width in each SAM phase.** The change in width Q95 − Q05 of the internal variability over the region, in the seasons with a positive SAM index (top) and with a negative one (bottom), 1993–2013 against hist-nat's whole record: the multi-model median (bars) and every model (markers).

In [ ]:
#(t): The change in width in each phase, averaged over REGION
phase_region_da = oc.regional_mean(sam_phase_width_ds['width_change'], REGION)
phase_region_da

In [ ]:
e4_block = partial(pp.forcing_bars, {f'Δ width, SAM > 0 ({UNITS})': phase_region_da.sel(phase='positive'),
                                     f'Δ width, SAM < 0 ({UNITS})': phase_region_da.sel(phase='negative')},
                   bars=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = e4_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'E4_width_by_sam_phase')

In [ ]:
e5_block = partial(pp.scatter_by_season, sam_change_da, width_region_da, 'Δ SAM index', f'Δ(Q95 − Q05) ({UNITS})',
                   experiments=EXPERIMENTS, columns=SEASONS, style=STYLE)
panels = e5_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'E5_sam_vs_width')

## F. Synthesis and additivity

**F1. Every forcing's change, and their sum.** Each experiment's change from its own 1850–1900 baseline, averaged over the region (notebook 03, sections 6 and 11): the mean (top) and the variance of the internal variability (bottom; variances add up where widths do not). The sum of the four single forcings stands next to historical: where the two differ, the forcings do not add up.

In [ ]:
#(t): Additivity of the mean (notebook 03, section 6) and of the variance (section 11)
mean_additivity_ds = rc.additivity(own_changes_da)
variance_additivity_ds = rc.additivity(own_variance_changes_da)
mean_additivity_ds

In [ ]:
#(t): Their terms averaged over REGION, on (model, term, season)
mean_terms_region_da = oc.regional_mean(mean_additivity_ds['response'], REGION)
variance_terms_region_da = oc.regional_mean(variance_additivity_ds['response'], REGION)

In [ ]:
F1_BARS = [*rc.ADDITIVE_PARTS, 'sum of parts', 'historical']
f1_block = partial(pp.forcing_bars, {f'Δ mean ({UNITS})': mean_terms_region_da,
                                     f'Δ variance ({UNITS}²)': variance_terms_region_da},
                   x_dim='term', bars=F1_BARS, columns=SEASONS, style=STYLE)
panels = f1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'F1_synthesis')

**F2. Where the forcings do not add up.** Multi-model median of historical's change minus the sum of the single forcings' changes, each from its own 1850–1900 baseline: the mean (top) and the variance of the internal variability (bottom). The residual mixes interaction between forcings, forcings left out (land use) and sampling noise.

In [ ]:
f2_mean_block = partial(pp.map_block, mean_additivity_ds['response'].sel(term='residual').median('model'),
                        scales.change, columns=SEASONS, title='Mean', style=STYLE)
f2_variance_block = partial(pp.map_block, variance_additivity_ds['response'].sel(term='residual').median('model'),
                            variance_scale, columns=SEASONS, title='Variance', style=STYLE)
panels = pp.compose([[f2_mean_block], [f2_variance_block]], heights=[2.6, 2.6], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'F2_additivity_residual')

## V. Verification against ERA5

The second link of the attribution chain: historical has to be consistent with ERA5 for the models' forced changes to be credible for the real world (Suarez-Gutierrez et al., 2021). Regional means come first, as the paper recommends, with the grid-point maps as the spatial detail. From notebook 02 (sections 5–9, and part 2).

**V1. Rank histograms.** ERA5's rank among each model's historical members (rows), regional means, 1979–2013, per season (columns), on ten bins of scaled rank (0: below every member, 1: above every member). Grey: the range a perfect model gives, each member in turn ranked against the others (5–95%). Flat bars inside the grey: ERA5 could be a member. A U shape: too little spread; a dome: too much; a slope: a different forced response. Corner: ERA5's percentile for the dispersion among the perfect-model values (✗ beyond their 5–95%). One figure each for historical's anomalies (signal and noise), historical's detrended anomalies (noise only) and hist-nat's anomalies (which ERA5 should *not* fit).

In [ ]:
V1_CASES = {'historical_anomaly': ('historical', 'anomaly'), 'historical_detrended': ('historical', 'detrended'),
            'hist-nat_anomaly': ('hist-nat', 'anomaly')}
v1_panels = {name: pp.rank_histograms(regional_ranks_ds.sel(region=REGION, experiment=experiment,
                                                             treatment=treatment),
                                      columns=SEASONS, style=STYLE, tag_start=0)
             for name, (experiment, treatment) in V1_CASES.items()}

In [ ]:
for name, panels in v1_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'V1_rank_histograms_{name}')

**V2. Where each model is adequate.** (Top) How many models are adequate at each grid point (the paper's Fig. 8): ERA5 below every member in fewer than 10% of years, above every member in fewer than 10%, and inside the members' central 75% in at most 80% of them. (Below) Each model: where ERA5 is beyond every member too often, above (red) or below (blue), and hatched where it is in the central 75% too often (too much variability).

In [ ]:
#(t): How many models are adequate at each grid point
adequate_count_ds = ev.adequate_count(spatial_table_ds)
adequate_count_ds

In [ ]:
#(t): Each model's edge class (evp.EDGE_CLASSES)
edge_class_da = evp.edge_class(spatial_table_ds)

In [ ]:
v2_count_block = partial(pp.map_block, adequate_count_ds['n_adequate'], fr.majority_scale(N_MODELS, 'Adequate models'),
                         columns=SEASONS, style=STYLE)
v2_edge_block = partial(pp.map_block, edge_class_da, pp.class_scale(evp.EDGE_CLASSES, evp.EDGE_COLORS,
                                                                    '% of years ERA5 is beyond every member'),
                        row_dim='model', columns=SEASONS, style=STYLE,
                        hatches=[pp.Hatch(spatial_table_ds['central_flag'] == 1, '////',
                                          "ERA5 in the members' central 75% in more than 80% of years")])
panels = pp.compose([[v2_count_block], [v2_edge_block]], heights=[2.6, 1.8 * N_MODELS + 1.0], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'V2_spatial_evaluation')

**V3. What is wrong, and is it the forcing or the variability?** Each model's diagnosis over 1979–2013 (left of the colour bar: adequate) and, for each problem, whether it is the same in 1979–1996 and 1997–2013 (the variability) or changes between them (the forced response).

In [ ]:
v3_diagnosis_block = partial(pp.map_block, spatial_table_ds['diagnosis'],
                             pp.class_scale(list(ev.DIAGNOSES.values()), evp.DIAGNOSIS_COLORS, 'Diagnosis'),
                             row_dim='model', columns=SEASONS, style=STYLE)
v3_period_block = partial(pp.map_block, spatial_table_ds['period_diagnosis'],
                          pp.class_scale(list(ev.PERIOD_DIAGNOSES.values()), evp.PERIOD_COLORS,
                                         'Early against late'),
                          row_dim='model', columns=SEASONS, style=STYLE)
panels = pp.compose([[v3_diagnosis_block], [v3_period_block]], heights=[1.8 * N_MODELS + 1.0] * 2, style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'V3_diagnosis')

**V4. Which experiments is the observed trend consistent with?** ERA5's 1979–2013 regional trend as a percentile of each experiment's members (columns), every model (rows) and season (panels). Outlined: outside the members' 5–95%. The attribution needs ERA5 inside historical and outside hist-nat.

In [ ]:
trend_consistency_ds = regional_consistency_ds.sel(region=REGION, statistic='trend')
trend_consistency_ds['percentile'].to_dataframe().unstack('season').round(0)

In [ ]:
v4_block = partial(pp.percentile_tables, trend_consistency_ds['percentile'], trend_consistency_ds['verdict'],
                   columns=['hist-nat', *EXPERIMENTS], panels=SEASONS, style=STYLE)
panels = v4_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'V4_trend_consistency')

**V5. The moment test over the whole map** (notebook 02, section 7), for the supplement as a table: ✗ where ERA5 is flagged over more of the map than a perfect model is 90% of the time.

In [ ]:
field_verdict_df = eval_map_table_ds['field_verdict'].sel(season=SEASONS).to_series().unstack(['statistic', 'season'])
field_verdict_df.replace({0.0: '✓', 1.0: '✗'})

## S. Supplementary

**S1. The SAM regression in every model.** The change in temperature with one unit of the SAM index, from internal variability alone (each member minus its ensemble's mean), every model (rows) and season.

In [ ]:
s1_block = partial(pp.map_block, sam_regression_da, regression_scale, row_dim='model', columns=SEASONS, style=STYLE)
panels = s1_block(tag_start=0)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'S1_sam_regression')

**S2. Every model's probability ratios.** As C3 for historical, every model (rows); dots: the 95% bootstrap range includes 1.

In [ ]:
#(t): historical's ratios in every model, and where they are not significant
model_ratio_da = log_ratio_da.sel(experiment='historical')
model_ratio_not_significant_da = ~pr_ds['pr_significant'].sel(experiment='historical', return_period=RETURN_PERIOD) \
    & model_ratio_da.notnull()

In [ ]:
s2_panels = {tail: pp.map_block(model_ratio_da.sel(tail=tail), ratio_scale, row_dim='model', columns=SEASONS,
                                hatches=[pp.Hatch(model_ratio_not_significant_da.sel(tail=tail), *pp.NOT_SIGNIFICANT)],
                                title=f'historical, {tail} seasons', style=STYLE, tag_start=0)
             for tail in ('warm', 'cold')}

In [ ]:
for tail, panels in s2_panels.items():
    fr.save(panels, CANDIDATE_DIR, f'S2_probability_ratios_every_model_{tail}')

## The candidate layouts

The blocks above, put together (`pp.compose`), the panel letters running on from block to block. A layout figure that is a single candidate is not drawn again:

| Layout | Fig. 1 | Fig. 2 | Fig. 3 | Fig. 4 | Fig. 5 |
|---|---|---|---|---|---|
| L1 current plan | A1 + C1 | B1 + C2 | D2 | E2 + E3 | |
| L2 merged mechanism | A1 + C1 | B1 + C2 | D2, D3 / E1, E2 | F1 | |
| L3 original outline | A2 (historical) | the width as A2 | A1 | B1 | F2 |
| L4 obs-led | A4 + A5 | A1 | B1 or B3 | F1 | D2 + E2 |
| L5 short NCC | the strongest result (a candidate: R1 + C2) | A1 or F1 | D2 + E3 | C2 | |

The figures with D2 need the sea ice, so they are in the (JASMIN) section.

**L1 and L2, Fig. 1: the mean change and its extremes.** A1 above C1.

In [ ]:
panels = pp.compose([[a1_block], [c1_block]], heights=[6.6, 3.2], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L1_fig1_mean_and_extremes')

**L1 and L2, Fig. 2: the width and what it does to the extremes.** B1 above C2.

In [ ]:
panels = pp.compose([[b1_block], [c2_block]], heights=[6.6, 3.2], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L1_fig2_width_and_extremes')

**L1, Fig. 4: the SAM.** E2 for historical above E3.

In [ ]:
panels = pp.compose([[e2_blocks['historical']], [e3_block]], heights=[5.0, 2.6], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L1_fig4_sam')

**L3, Fig. 2: the width, as A2.** For historical: the median change in width, and how many models agree on its sign.

In [ ]:
#(t): How many models agree on the sign of the median change in width
width_sign_count_da = (width_agreement_ds['sign_agreement'] * width_agreement_ds['n_models']).round()

In [ ]:
l3_median_block = partial(pp.map_block, width_agreement_ds['median'].sel(experiment='historical'),
                          scales.spread_change, columns=SEASONS, style=STYLE)
l3_count_block = partial(pp.map_block, width_sign_count_da.sel(experiment='historical'), count_scale,
                         columns=SEASONS, style=STYLE)
panels = pp.compose([[l3_median_block], [l3_count_block]], heights=[2.5, 2.5], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L3_fig2_width_historical')

**L4, Fig. 1: the observed change.** A4 above A5.

In [ ]:
panels = pp.compose([[a4_trend_block], [a4_percentile_block], [a5_block]], heights=[4.4, 2.6, 2.4], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L4_fig1_observed')

**L5, Fig. 1: a candidate headline.** The whole distributions over the region (R1) above what that does to the extremes (C2).

In [ ]:
panels = pp.compose([[r1_blocks[REGION]], [c2_block]], heights=[3.0, 3.2], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L5_fig1_distributions_and_extremes')

### The layouts with sea ice (JASMIN)

**L2, Fig. 3: the mechanisms.** The sea-ice row (D2, D3) above the SAM row (E1, E2 for historical).

In [ ]:
panels = pp.compose([[d2_block], [d3_block], [e1_block], [e2_blocks['historical']]],
                    heights=[2.6, 2.4, 2.4, 5.0], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L2_fig3_mechanisms')

**L4, Fig. 5: the mechanisms, short.** D2 above E2 for historical.

In [ ]:
panels = pp.compose([[d2_block], [e2_blocks['historical']]], heights=[2.6, 5.0], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L4_fig5_mechanisms')

**L5, Fig. 3: the mechanisms, as scatter.** D2 above E3.

In [ ]:
panels = pp.compose([[d2_block], [e3_block]], heights=[2.6, 2.6], style=STYLE)

In [ ]:
fr.save(panels, CANDIDATE_DIR, 'L5_fig3_mechanisms')

## What was saved

In [ ]:
pd.Series(sorted(path.name for path in CANDIDATE_DIR.glob('*.pdf')), name=str(CANDIDATE_DIR)).to_frame()